In [5]:
import os

from eoh.routefinder.problem_class import CeohRoutefinderProblem
from routefinder.util.data_creation import build_routing_model, create_dataset, ORTOOLS_SCALING_FACTOR

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

import numpy as np

from dotenv import load_dotenv

In [6]:
# OR-Tools

def run_or_tools_best_first(data, first_solution_strategy):

    manager, routing = build_routing_model(data)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = first_solution_strategy
    search_parameters.time_limit.seconds = 600
    search_parameters.solution_limit = 1
    search_parameters.log_search = False

    final_solution = routing.SolveWithParameters(search_parameters)

    if final_solution is None:
        print("No Solution found!")
        return None
    else:
        objective = final_solution.ObjectiveValue()
        return objective


In [9]:
# Run experiments and add solutions to npz files

# Configuration
# load BASE_PATH from .env file
load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")

#problem_name ="ovrpmbltw"
problem_name = "cvrp"

instance_class = "test"
instance_file = "100"

# Build Problem and load datasets

problem = CeohRoutefinderProblem(problem_name)

final_data = dict(problem.extract_npz(instance_class, instance_file))

datasets = problem.get_datasets_for_file(instance_class, instance_file)


# All OR Tools Strategies
strategies = [
    routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC,
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC,
    routing_enums_pb2.FirstSolutionStrategy.PATH_MOST_CONSTRAINED_ARC,
    routing_enums_pb2.FirstSolutionStrategy.SAVINGS,
    routing_enums_pb2.FirstSolutionStrategy.CHRISTOFIDES,
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION,
    routing_enums_pb2.FirstSolutionStrategy.LOCAL_CHEAPEST_INSERTION,
    routing_enums_pb2.FirstSolutionStrategy.GLOBAL_CHEAPEST_ARC,
    routing_enums_pb2.FirstSolutionStrategy.LOCAL_CHEAPEST_ARC,
]

# Runtime issue?
# routing_enums_pb2.FirstSolutionStrategy.FIRST_UNBOUND_MIN_VALUE

strategy_names = {v: k for k, v in routing_enums_pb2.FirstSolutionStrategy.__dict__.items() if isinstance(v, int)}

# For each strategy, run all npz instances

for strategy in strategies:
    print("="*40)
    print(f"Testing strategy: {strategy_names[strategy]}")
    final_data[f"{strategy_names[strategy]}_sol"] = []
    for instance in datasets:
        solution = run_or_tools_best_first(instance, strategy)
        if solution is not None:
            solution /= ORTOOLS_SCALING_FACTOR
        final_data[f"{strategy_names[strategy]}_sol"].append(solution)

print(final_data)

np.savez(
    os.path.join(
        os.environ["BASE_PATH"], 'vrp_instances', problem_name, instance_class, f"{instance_file}_with_or_tools_first_solution_sols.npz"
    ),
     **final_data)  # data is a dict here


Testing strategy: AUTOMATIC
Testing strategy: PATH_CHEAPEST_ARC
Testing strategy: PATH_MOST_CONSTRAINED_ARC
Testing strategy: SAVINGS
Testing strategy: CHRISTOFIDES
Testing strategy: PARALLEL_CHEAPEST_INSERTION
Testing strategy: LOCAL_CHEAPEST_INSERTION
Testing strategy: GLOBAL_CHEAPEST_ARC
Testing strategy: LOCAL_CHEAPEST_ARC
{'actions': array([[96, 70, 54, ...,  0,  0,  0],
       [ 1, 99,  0, ..., 87,  0,  0],
       [23, 96, 89, ...,  0,  0,  0],
       ...,
       [97,  8,  5, ...,  0,  0,  0],
       [31, 72, 27, ...,  0,  0,  0],
       [89, 80, 38, ...,  0,  0,  0]], dtype=int64), 'costs': array([-15.30685, -17.68449, -17.42872, -17.83903, -15.88052, -15.29224,
       -14.08795, -15.33241, -17.52107, -17.44387, -18.19163, -18.0406 ,
       -17.83526, -12.74163, -17.86126, -17.04799, -18.34873, -17.16195,
       -16.14541, -16.73222, -17.27654, -17.36606, -14.12795, -17.06433,
       -19.75954, -19.26269, -14.65919, -14.06985, -20.37995, -18.19082,
       -20.14849, -17.55407, -

In [8]:
npz_sol = np.load(os.path.join(os.environ["BASE_PATH"], 'vrp_instances', problem_name, instance_class, f"{instance_file}_with_or_tools_first_solution_sols.npz"), allow_pickle=True)

solutions = dict(npz_sol)

print(solutions.keys())



dict_keys(['actions', 'costs', 'time', 'AUTOMATIC_sol', 'PATH_CHEAPEST_ARC_sol', 'PATH_MOST_CONSTRAINED_ARC_sol', 'SAVINGS_sol', 'CHRISTOFIDES_sol', 'PARALLEL_CHEAPEST_INSERTION_sol', 'LOCAL_CHEAPEST_INSERTION_sol', 'GLOBAL_CHEAPEST_ARC_sol', 'LOCAL_CHEAPEST_ARC_sol'])
